In [2]:
import pandas as pd
import sys
import os
import numpy as np
import tensorflow as tf

sys.path.append(os.path.abspath("../src"))

import my_utils  # noqa: F401

from gensim.models import Word2Vec
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
w2v_model = Word2Vec.load("../data/word2vec.model")
wv = w2v_model.wv
wv.vectors.shape

(47014, 200)

In [4]:
labeled_df = pd.read_csv('../data/labeledTrainData.tsv.zip', header=0, delimiter="\t", quoting=3)
labeled_df.head()

,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."
3,"""3630_4""",0,"""It must be assumed that those who praised thi..."
4,"""9495_8""",1,"""Superbly trashy and wondrously unpretentious ..."


In [5]:
wordlist = labeled_df.review.nlp.review_to_wordlist()
wordlist

0        [stuff, going, moment, mj, started, listening,...
1        [classic, war, worlds, timothy, hines, enterta...
2        [film, starts, manager, nicholas, bell, giving...
3        [must, assumed, praised, film, greatest, filme...
4        [superbly, trashy, wondrously, unpretentious, ...
                               ...                        
24995    [seems, like, consideration, gone, imdb, revie...
24996    [believe, made, film, completely, unnecessary,...
24997    [guy, loser, get, girls, needs, build, picked,...
24998    [minute, documentary, bu, uel, made, early, on...
24999    [saw, movie, child, broke, heart, story, unfin...
Name: review, Length: 25000, dtype: object

### What's the max. count of tokens in review?

In [6]:
#max_sequence_len = max([len(lst) for lst in wordlist])
max_sequence_len = 400
max_sequence_len

400

In [7]:
X_embedded = []

for lst in wordlist:
    embedding = []
    for word in lst:
        if word in wv:
            embedding.append(wv[word])
        else:
            embedding.append(np.zeros(wv.vector_size))
    X_embedded.append(embedding)

In [8]:
padded_sequences = pad_sequences(
    sequences=X_embedded, 
    maxlen=max_sequence_len, 
    padding='post', 
    truncating='post'
)

padded_sequences.shape

(25000, 400, 200)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(max_sequence_len, wv.vector_size)),
    tf.keras.layers.Masking(mask_value=0.0),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ masking_2 (Masking)             │ (None, 400, 200)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       168,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 168,577 (658.50 KB)

 Trainable params: 168,577 (658.50 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.fit(
    padded_sequences, 
    labeled_df.sentiment.values, 
    epochs=5, 
    batch_size=32, 
    validation_split=0.2, 
    shuffle=True
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 77s 121ms/step - accuracy: 0.5404 - loss: 0.6888 - val_accuracy: 0.5296 - val_loss: 0.6891
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 114s 182ms/step - accuracy: 0.5436 - loss: 0.6859 - val_accuracy: 0.5298 - val_loss: 0.6891
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 104s 167ms/step - accuracy: 0.5430 - loss: 0.6849 - val_accuracy: 0.5296 - val_loss: 0.6915
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 87s 139ms/step - accuracy: 0.5459 - loss: 0.6843 - val_accuracy: 0.5312 - val_loss: 0.6907
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 119s 191ms/step - accuracy: 0.5457 - loss: 0.6839 - val_accuracy: 0.5278 - val_loss: 0.6904
